# DeXposure GraphPFN Link Prediction Experiment (Plan-Driven)

This notebook follows `experiment-plan.md` to run the full experiment:
- Main task: edge existence + edge weight (t -> t+1)
- Auxiliary task: node TVL log-change (t -> t+1)
- Core comparison: frozen-encoder probing vs finetuned GraphPFN
- Negative sampling: uniform 5:1

Outputs (saved to disk):
- `data_quality.json`
- `metrics.json`
- `predictions_edges_test.csv`
- `predictions_nodes_test.csv`

In [ ]:
import os
import sys
from pathlib import Path

GRAPHPFN_ROOT = Path.cwd()
if GRAPHPFN_ROOT.name == "notebooks":
    GRAPHPFN_ROOT = GRAPHPFN_ROOT.parent
os.chdir(GRAPHPFN_ROOT)
sys.path.insert(0, str(GRAPHPFN_ROOT))

print("Working directory:", GRAPHPFN_ROOT)

In [ ]:
import json
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score, 
    roc_auc_score, 
    mean_absolute_error, 
    mean_squared_error,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

from lib.graphpfn.model import GraphPFN, GraphPFNLayerWrapper
from lib.limix.model.layer import MultiheadAttention as MHA
from lib.util import TaskType

# 设置matplotlib显示中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
DATA_PATH = "data/historical-network_week_2025-07-01.json"
META_PATH = "data/meta_df.csv"
OUTPUT_DIR = "output/dexposure_graphpfn_link"
CHECKPOINT_PATH = "checkpoints/graphpfn-v1.ckpt"

NEG_RATIO = 5
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

EDGE_BATCH_SIZE = 20000  # set None to score all pairs at once
HIDDEN_DIM = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 5

EXIST_LOSS_WEIGHT = 1.0
WEIGHT_LOSS_WEIGHT = 1.0
NODE_LOSS_WEIGHT = 0.5

FINETUNE_FULL = False  # True = unfreeze all GraphPFN params
MAX_WEEKS = None  # set an int for quick debug runs

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 配置说明

本实验遵循 `experiment-plan.md` 规范，主要配置如下：

### 训练配置
- `NEG_RATIO=5`: 负采样比例（uniform sampling，neg:pos = 5:1）
- `TRAIN_RATIO=0.70`, `VAL_RATIO=0.15`: 时间序列切分（train 70% / val 15% / test 15%）
- `EPOCHS=5`: 训练轮数
- `LR=1e-3`, `WEIGHT_DECAY=1e-4`: 优化器参数

### 损失权重
- `EXIST_LOSS_WEIGHT=1.0`: Edge existence 分类损失
- `WEIGHT_LOSS_WEIGHT=1.0`: Edge weight 回归损失（仅正例）
- `NODE_LOSS_WEIGHT=0.5`: Node TVL log-change 辅助损失

### 模型配置
- `HIDDEN_DIM=256`: Link Scorer 和 Node Head 的隐藏层维度
- `FINETUNE_FULL=False`: 是否微调整个GraphPFN encoder（False表示仅训练scorer）
- `MAX_WEEKS=None`: 限制训练周数（用于调试，None表示使用全部数据）

### 对比实验
1. **Frozen-encoder probing**: GraphPFN encoder冻结，仅训练scorer heads
2. **Finetuned encoder**: 整个模型端到端训练

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 元数据处理

加载 `meta_df.csv` 获取protocol的category信息：
- 构建category到索引的映射（用于one-hot编码）
- 处理缺失category为"Unknown"
- 为每个protocol提供category信息

## 数据加载

从历史weekly network JSON文件中加载数据。文件较大时使用ijson流式读取。

数据格式：
```json
{
  "data": {
    "2022-01-03": {
      "nodes": [{"id": "...", "size": ..., "composition": {...}}],
      "links": [{"source": "...", "target": "...", "size": ...}]
    },
    ...
  }
}
```

## 特征工程与图构建

根据 `experiment-plan.md` 规范，对每个snapshot构建节点特征和图结构：

### 节点特征（MVP）
1. **log_size**: `log1p(TVL)` - 总价值锁定
2. **num_tokens**: composition中的token数量
3. **max_share**: 最大token占比（集中度指标）
4. **entropy**: token分布的熵（多样性指标）
5. **category**: one-hot编码（来自meta_df）

### 图构建与清洗
- **保留**: `target != null` 的边
- **丢弃**: source或target不在当周节点集合的边
- **统计**: 目标为null的边比例、端点缺失的边比例

### 数据质量指标
每个snapshot记录：
- 节点数、边数
- 数据清洗丢弃比例
- 与下一周的节点重叠率
- category缺失比例

## 训练样本构造

将周对 (t → t+1) 转换为训练样本。

### 正例边（Positive）
- t+1周实际存在的边
- 只保留两端点都在t周节点集合中的边
- 标签：`y_exist=1`, `y_weight=log1p(size_{t+1})`

### 负例边（Negative）
- 从t周节点集合 V_t × V_t 中均匀采样
- **负采样比例**: neg:pos = 5:1（可配置）
- 排除已存在的正例边
- 固定随机种子保证可复现
- 标签：`y_exist=0`, `y_weight=0`（masked out）

### 辅助任务：Node TVL预测
- 只对 t 和 t+1 周都存在的节点计算
- `y_node = log1p(TVL_{t+1}) - log1p(TVL_t)`
- 使用 `node_mask` 标记有效节点

### 样本组织
- 正负例混合并shuffle
- 每个样本包含：
  - 节点特征 (t周)
  - 图结构 (t周)
  - Edge pairs (u, v) + 标签
  - Node labels (TVL change) + mask

## 数据集切分

采用时间序列切分（避免未来信息泄漏）：
- **Train**: 前70%的周对
- **Validation**: 中间15%的周对
- **Test**: 最后15%的周对

这种切分方式符合实际预测场景（用过去预测未来）。

In [ ]:
meta_df = pd.read_csv(META_PATH)
meta_df["id"] = meta_df["id"].astype(str)

category_list = sorted(meta_df["category"].dropna().unique().tolist())
if "Unknown" not in category_list:
    category_list.append("Unknown")

category_to_idx = {c: i for i, c in enumerate(category_list)}
meta_category = meta_df.set_index("id")["category"].to_dict()

print("Categories:", len(category_list))

In [ ]:
def load_network_data(path: str) -> Dict:
    path = Path(path)
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"Loading {path} ({size_mb:.1f} MB)")

    with path.open("rb") as f:
        try:
            import ijson
            data = {k: v for k, v in ijson.kvitems(f, "data")}
            return data
        except Exception:
            f.seek(0)
            payload = json.load(f)
            return payload["data"]

In [ ]:
EPS = 1e-12


def one_hot_category(category: str) -> np.ndarray:
    idx = category_to_idx.get(category, category_to_idx["Unknown"])
    vec = np.zeros(len(category_list), dtype=np.float32)
    vec[idx] = 1.0
    return vec


def node_features(node: Dict) -> Tuple[np.ndarray, float, str]:
    node_id = str(node.get("id"))
    size = float(node.get("size", 0.0))
    comp = node.get("composition", {}) or {}

    log_size = math.log1p(max(size, 0.0))
    num_tokens = float(len(comp))

    if size > 0 and comp:
        values = np.array(list(comp.values()), dtype=np.float64)
        values = np.maximum(values, 0.0)
        total = values.sum() + EPS
        shares = values / total
        max_share = float(shares.max())
        entropy = float(-(shares * np.log(shares + EPS)).sum())
    else:
        max_share = 0.0
        entropy = 0.0

    category = meta_category.get(node_id, "Unknown")
    cat_vec = one_hot_category(category)

    feats = np.array([log_size, num_tokens, max_share, entropy], dtype=np.float32)
    feats = np.concatenate([feats, cat_vec], axis=0)

    return feats, size, category


def build_snapshot(date: str, snapshot: Dict) -> Dict:
    nodes = snapshot.get("nodes", [])
    links = snapshot.get("links", [])

    node_ids = []
    features = []
    sizes = []
    categories = []

    for node in nodes:
        node_id = node.get("id")
        if node_id is None:
            continue
        feats, size, category = node_features(node)
        node_ids.append(str(node_id))
        features.append(feats)
        sizes.append(size)
        categories.append(category)

    id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

    null_target = 0
    missing_endpoint = 0
    src = []
    dst = []
    weights = []

    for link in links:
        source = link.get("source")
        target = link.get("target")

        if target is None:
            null_target += 1
            continue
        if source is None:
            missing_endpoint += 1
            continue

        source = str(source)
        target = str(target)

        if source not in id_to_idx or target not in id_to_idx:
            missing_endpoint += 1
            continue

        src.append(id_to_idx[source])
        dst.append(id_to_idx[target])
        weights.append(float(link.get("size", 0.0)))

    num_links = len(links)
    pct_target_null = null_target / max(num_links, 1)
    pct_endpoint_missing = missing_endpoint / max(num_links, 1)
    pct_category_unknown = categories.count("Unknown") / max(len(categories), 1)

    return {
        "date": date,
        "node_ids": node_ids,
        "features": np.array(features, dtype=np.float32),
        "sizes": np.array(sizes, dtype=np.float32),
        "categories": categories,
        "edge_src": np.array(src, dtype=np.int64),
        "edge_dst": np.array(dst, dtype=np.int64),
        "edge_weight": np.array(weights, dtype=np.float32),
        "num_links_raw": num_links,
        "num_edges": len(src),
        "pct_target_null_dropped": pct_target_null,
        "pct_endpoint_missing_dropped": pct_endpoint_missing,
        "pct_category_unknown": pct_category_unknown,
    }


def snapshot_to_dgl(snapshot: Dict) -> dgl.DGLGraph:
    graph = dgl.graph(
        (snapshot["edge_src"], snapshot["edge_dst"]),
        num_nodes=len(snapshot["node_ids"]),
    )
    if len(snapshot["edge_weight"]) > 0:
        graph.edata["weight"] = torch.tensor(snapshot["edge_weight"], dtype=torch.float32)
    return graph

In [ ]:
network_data = load_network_data(DATA_PATH)
all_dates = sorted(network_data.keys())
if MAX_WEEKS is not None:
    all_dates = all_dates[:MAX_WEEKS]

snapshots = []
for date in all_dates:
    snapshots.append(build_snapshot(date, network_data[date]))

for i in range(len(snapshots) - 1):
    current_nodes = set(snapshots[i]["node_ids"])
    next_nodes = set(snapshots[i + 1]["node_ids"])
    overlap_ratio = len(current_nodes & next_nodes) / max(len(current_nodes), 1)
    snapshots[i]["overlap_ratio_next_week"] = overlap_ratio

if snapshots:
    snapshots[-1]["overlap_ratio_next_week"] = None

print(f"Loaded {len(snapshots)} snapshots")

In [ ]:
@dataclass
class WeekPair:
    time_t: str
    time_t1: str
    node_ids: List[str]
    categories: List[str]
    sizes_t: np.ndarray
    features_t: np.ndarray
    edge_src_t: np.ndarray
    edge_dst_t: np.ndarray
    pair_src: np.ndarray
    pair_dst: np.ndarray
    y_exist: np.ndarray
    y_weight: np.ndarray
    weight_mask: np.ndarray
    y_node: np.ndarray
    node_mask: np.ndarray
    pos_edge_count: int
    neg_edge_count: int


def sample_negatives(num_nodes: int, pos_set: set, num_neg: int, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    neg_src = []
    neg_dst = []
    seen = set(pos_set)
    while len(neg_src) < num_neg:
        u = int(rng.integers(0, num_nodes))
        v = int(rng.integers(0, num_nodes))
        if (u, v) in seen:
            continue
        seen.add((u, v))
        neg_src.append(u)
        neg_dst.append(v)
    return np.array(neg_src, dtype=np.int64), np.array(neg_dst, dtype=np.int64)


def build_week_pairs(snapshots: List[Dict], neg_ratio: int, seed: int) -> List[WeekPair]:
    rng = np.random.default_rng(seed)
    pairs = []

    for t in range(len(snapshots) - 1):
        snap_t = snapshots[t]
        snap_t1 = snapshots[t + 1]

        id_to_idx = {nid: i for i, nid in enumerate(snap_t["node_ids"])}
        size_t1_map = {nid: size for nid, size in zip(snap_t1["node_ids"], snap_t1["sizes"])}

        pos_src = []
        pos_dst = []
        pos_w = []
        pos_set = set()

        for src_idx, dst_idx, w in zip(snap_t1["edge_src"], snap_t1["edge_dst"], snap_t1["edge_weight"]):
            src_id = snap_t1["node_ids"][int(src_idx)]
            dst_id = snap_t1["node_ids"][int(dst_idx)]
            if src_id not in id_to_idx or dst_id not in id_to_idx:
                continue
            u = id_to_idx[src_id]
            v = id_to_idx[dst_id]
            pos_src.append(u)
            pos_dst.append(v)
            pos_w.append(math.log1p(max(w, 0.0)))
            pos_set.add((u, v))

        pos_src = np.array(pos_src, dtype=np.int64)
        pos_dst = np.array(pos_dst, dtype=np.int64)
        pos_w = np.array(pos_w, dtype=np.float32)

        num_pos = len(pos_src)
        num_neg = num_pos * neg_ratio
        neg_src, neg_dst = sample_negatives(len(snap_t["node_ids"]), pos_set, num_neg, rng)

        pair_src = np.concatenate([pos_src, neg_src])
        pair_dst = np.concatenate([pos_dst, neg_dst])
        y_exist = np.concatenate([np.ones(num_pos, dtype=np.float32), np.zeros(num_neg, dtype=np.float32)])
        y_weight = np.concatenate([pos_w, np.zeros(num_neg, dtype=np.float32)])
        weight_mask = np.concatenate([np.ones(num_pos, dtype=np.float32), np.zeros(num_neg, dtype=np.float32)])

        order = rng.permutation(len(pair_src))
        pair_src = pair_src[order]
        pair_dst = pair_dst[order]
        y_exist = y_exist[order]
        y_weight = y_weight[order]
        weight_mask = weight_mask[order]

        y_node = np.zeros(len(snap_t["node_ids"]), dtype=np.float32)
        node_mask = np.zeros(len(snap_t["node_ids"]), dtype=bool)
        for i, nid in enumerate(snap_t["node_ids"]):
            if nid in size_t1_map:
                y_node[i] = math.log1p(max(size_t1_map[nid], 0.0)) - math.log1p(max(snap_t["sizes"][i], 0.0))
                node_mask[i] = True

        pairs.append(
            WeekPair(
                time_t=snap_t["date"],
                time_t1=snap_t1["date"],
                node_ids=snap_t["node_ids"],
                categories=snap_t["categories"],
                sizes_t=snap_t["sizes"],
                features_t=snap_t["features"],
                edge_src_t=snap_t["edge_src"],
                edge_dst_t=snap_t["edge_dst"],
                pair_src=pair_src,
                pair_dst=pair_dst,
                y_exist=y_exist,
                y_weight=y_weight,
                weight_mask=weight_mask,
                y_node=y_node,
                node_mask=node_mask,
                pos_edge_count=num_pos,
                neg_edge_count=num_neg,
            )
        )

    return pairs

In [ ]:
week_pairs = build_week_pairs(snapshots, NEG_RATIO, SEED)
print(f"Week pairs: {len(week_pairs)}")
if week_pairs:
    print("Example pair:", week_pairs[0].time_t, "->", week_pairs[0].time_t1)
    print("  Nodes:", len(week_pairs[0].node_ids))
    print("  Pos edges:", week_pairs[0].pos_edge_count)
    print("  Neg edges:", week_pairs[0].neg_edge_count)

In [ ]:
def split_week_pairs(pairs: List[WeekPair], train_ratio: float, val_ratio: float):
    n = len(pairs)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    train_pairs = pairs[:train_end]
    val_pairs = pairs[train_end:val_end]
    test_pairs = pairs[val_end:]
    return train_pairs, val_pairs, test_pairs


train_pairs, val_pairs, test_pairs = split_week_pairs(week_pairs, TRAIN_RATIO, VAL_RATIO)
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")

## Model: GraphPFN Encoder + Link Scorer + Node Head

GraphPFN is used as a node encoder. We pass dummy labels to satisfy the API and avoid label leakage.

In [ ]:
def graphpfn_encode(
    model: GraphPFN,
    graph: dgl.DGLGraph,
    features: torch.Tensor,
    device: torch.device,
    train_mask: Optional[torch.Tensor] = None,
    y_train: Optional[torch.Tensor] = None,
    task_type: TaskType = TaskType.REGRESSION,
    checkpointing: bool = True,
    batched_attn: bool = False,
) -> torch.Tensor:
    num_nodes = graph.num_nodes()

    if train_mask is None:
        train_mask = torch.ones(num_nodes, dtype=torch.bool, device=device)
    if y_train is None:
        y_train = torch.zeros(int(train_mask.sum().item()), device=device)

    features = features.to(device)
    y_train = y_train.to(device).float()
    train_mask = train_mask.to(device)
    graph = graph.to(device)

    tfm_features = torch.cat(
        [features[train_mask, ...], features[~train_mask, ...]],
        dim=0,
    )

    for module in model.modules():
        if isinstance(module, GraphPFNLayerWrapper):
            module.train_mask = train_mask
            module.graph = graph
        if isinstance(module, MHA):
            module.batched = batched_attn

    if device.type == "cpu":
        sdpa_backends = [torch.nn.attention.SDPBackend.MATH]
    elif batched_attn:
        sdpa_backends = [torch.nn.attention.SDPBackend.MATH]
    else:
        sdpa_backends = [
            torch.nn.attention.SDPBackend.FLASH_ATTENTION,
            torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION,
        ]

    with torch.nn.attention.sdpa_kernel(sdpa_backends):
        out = model.tfm.forward(
            x=tfm_features.unsqueeze(0),
            y=y_train.unsqueeze(0),
            eval_pos=y_train.shape[0],
            task_type="reg" if (task_type == TaskType.REGRESSION) else "cls",
            checkpointing=checkpointing,
        )

    inv_order = torch.argsort((~train_mask).float(), stable=True)
    order = torch.argsort(inv_order, stable=True)

    encoder_embed = out["encoder_embed"].squeeze(0)[order, ...]
    return encoder_embed


class LinkScorer(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int):
        super().__init__()
        in_dim = 4 * embed_dim
        self.exist_head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.weight_head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, h: torch.Tensor, src: torch.Tensor, dst: torch.Tensor):
        h_u = h[src]
        h_v = h[dst]
        z = torch.cat([h_u, h_v, h_u * h_v, (h_u - h_v).abs()], dim=-1)
        exist_logits = self.exist_head(z).squeeze(-1)
        weight_pred = self.weight_head(z).squeeze(-1)
        return exist_logits, weight_pred


class NodeHead(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h).squeeze(-1)


class GraphPFNLinkPredictor(nn.Module):
    def __init__(self, encoder: GraphPFN, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.encoder = encoder
        self.link_scorer = LinkScorer(embed_dim, hidden_dim)
        self.node_head = NodeHead(embed_dim, hidden_dim)

    def encode(self, graph: dgl.DGLGraph, features: torch.Tensor, device: torch.device) -> torch.Tensor:
        return graphpfn_encode(self.encoder, graph, features, device=device)

## 模型架构

本实验采用 GraphPFN 作为节点编码器，配合自定义的 Link Scorer 和 Node Head。

### GraphPFN Encoder（预训练）
- 输入：图结构 + 节点特征
- 输出：节点嵌入 h_u ∈ R^d
- 权重来源：预训练的 GraphPFN checkpoint
- 可选：冻结（probing）或微调（finetuned）

### Link Scorer（预测t+1周的边）
对任意节点对 (u, v)：
1. 构造pair特征：`z = [h_u, h_v, h_u * h_v, |h_u - h_v|]`
2. **Existence Head**: `p(exist) = sigmoid(MLP_exist(z))`
3. **Weight Head**: `w_pred = MLP_weight(z)` (预测log1p规模)

### Node Head（辅助任务）
- 输入：节点嵌入 h_u
- 输出：`TVL_log_change = MLP_node(h_u)`
- 目的：辅助学习更好的节点表示

### 训练策略
- **Multi-task learning**: 同时优化edge existence、edge weight和node TVL三个任务
- **Loss weights**: 1.0 * L_exist + 1.0 * L_weight + 0.5 * L_node

In [ ]:
def load_graphpfn_encoder(checkpoint_path: str, device: torch.device) -> GraphPFN:
    model = GraphPFN(edge_head=False)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model"], strict=False)
    model.to(device)
    return model


encoder_frozen = load_graphpfn_encoder(CHECKPOINT_PATH, DEVICE)
encoder_finetune = load_graphpfn_encoder(CHECKPOINT_PATH, DEVICE)

embed_dim = encoder_finetune.tfm.embed_dim
print("GraphPFN embed_dim:", embed_dim)

## 加载预训练GraphPFN

从checkpoint加载预训练的GraphPFN模型：
- 使用两个独立的encoder副本（一个frozen，一个可微调）
- 避免实验之间的相互干扰
- 获取embedding维度用于构建后续的scorer heads

In [ ]:
def set_encoder_trainable(model: GraphPFN, trainable: bool):
    for p in model.parameters():
        p.requires_grad = trainable

## Encoder权重控制

辅助函数：设置GraphPFN encoder的参数是否可训练。

- **trainable=False**: 冻结encoder（用于frozen-encoder probing）
- **trainable=True**: 允许encoder参数更新（用于finetuning）

In [ ]:
def iter_edge_batches(num_pairs: int, batch_size: Optional[int], rng: Optional[np.random.Generator] = None, shuffle: bool = True):
    """迭代edge batches，用于大规模图边级别的训练"""
    idx = np.arange(num_pairs)
    if shuffle and rng is not None:
        rng.shuffle(idx)
    if batch_size is None:
        yield idx
    else:
        for start in range(0, num_pairs, batch_size):
            yield idx[start : start + batch_size]


def train_one_epoch(
    model: GraphPFNLinkPredictor,
    pairs: List[WeekPair],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    edge_batch_size: Optional[int],
    finetune_encoder: bool,
) -> Dict[str, float]:
    """
    训练一个epoch
    
    损失函数组成：
    - L_exist: BCEWithLogitsLoss on all pairs (pos+neg)
    - L_weight: SmoothL1 on positive pairs only
    - L_node: SmoothL1 on nodes with node_mask=True
    
    总损失: L = 1.0*L_exist + 1.0*L_weight + 0.5*L_node
    """
    model.train()
    
    # 设置encoder的mode
    if finetune_encoder:
        model.encoder.train()
    else:
        model.encoder.eval()
    
    rng = np.random.default_rng(SEED)

    total_exist = 0.0
    total_weight = 0.0
    total_node = 0.0
    total_samples = 0

    for sample in pairs:
        # 构建图并获取节点embeddings
        graph = dgl.graph((sample.edge_src_t, sample.edge_dst_t), num_nodes=len(sample.node_ids))
        features = torch.tensor(sample.features_t, dtype=torch.float32)

        if finetune_encoder:
            h = model.encode(graph, features, device=device)
        else:
            with torch.no_grad():
                h = model.encode(graph, features, device=device)
            h = h.detach()

        # 计算node loss（针对整个图，只计算一次）
        node_true = torch.tensor(sample.y_node, dtype=torch.float32, device=device)
        node_mask = torch.tensor(sample.node_mask, dtype=torch.bool, device=device)
        node_pred = model.node_head(h)
        if node_mask.any():
            node_loss = F.smooth_l1_loss(node_pred[node_mask], node_true[node_mask])
        else:
            node_loss = torch.tensor(0.0, device=device)

        # 准备edge pairs
        src = torch.tensor(sample.pair_src, dtype=torch.long, device=device)
        dst = torch.tensor(sample.pair_dst, dtype=torch.long, device=device)
        y_exist = torch.tensor(sample.y_exist, dtype=torch.float32, device=device)
        y_weight = torch.tensor(sample.y_weight, dtype=torch.float32, device=device)
        weight_mask = torch.tensor(sample.weight_mask, dtype=torch.float32, device=device)

        optimizer.zero_grad()
        batches = list(iter_edge_batches(len(src), edge_batch_size, rng=rng, shuffle=True))

        # Edge-level训练：分batch处理所有pairs
        for b, batch_idx in enumerate(batches):
            logits, w_pred = model.link_scorer(h, src[batch_idx], dst[batch_idx])
            
            # Existence loss（所有pairs）
            exist_loss = F.binary_cross_entropy_with_logits(logits, y_exist[batch_idx])

            # Weight loss（仅正例pairs）
            mask = weight_mask[batch_idx] > 0.5
            if mask.any():
                weight_loss = F.smooth_l1_loss(w_pred[mask], y_weight[batch_idx][mask])
            else:
                weight_loss = torch.tensor(0.0, device=device)

            # 累积edge losses
            loss = (EXIST_LOSS_WEIGHT * exist_loss) + (WEIGHT_LOSS_WEIGHT * weight_loss)
            
            # 反向传播（保留计算图以便后续batches使用）
            retain = finetune_encoder and (b < len(batches) - 1)
            loss.backward(retain_graph=retain)

            total_exist += exist_loss.item()
            total_weight += weight_loss.item()

        # 在所有edge batches之后，单独处理node loss的反向传播
        if NODE_LOSS_WEIGHT > 0:
            (NODE_LOSS_WEIGHT * node_loss).backward()

        total_node += node_loss.item()
        total_samples += 1

        # 更新参数
        optimizer.step()

    return {
        "exist_loss": total_exist / max(total_samples, 1),
        "weight_loss": total_weight / max(total_samples, 1),
        "node_loss": total_node / max(total_samples, 1),
    }

## 模型评估

在训练集、验证集和测试集上评估两个模型：

**评估内容**：
1. **Frozen-encoder probing**（预训练表示的直接性能）
2. **Finetuned encoder**（端到端微调后的性能）

**输出**：
- 三个split的完整metrics
- 用于对比分析zero-shot vs finetuned的效果差异
- 识别是否预训练表示足够好（不需要微调）

In [ ]:
# 性能对比分析
print("=" * 80)
print("实验结果对比：Frozen-Encoder Probing vs Finetuned Encoder")
print("=" * 80)

def format_metric(metrics, split, metric_path, default="N/A"):
    """格式化指标输出"""
    try:
        val = metrics[split]
        for key in metric_path:
            val = val[key]
        if isinstance(val, float):
            return f"{val:.4f}"
        return str(val)
    except (KeyError, TypeError):
        return default

# 对比表格
comparison = []
for split in ["train", "val", "test"]:
    row = {"Split": split.upper()}
    
    # Edge Existence
    row["Probing AUPRC"] = format_metric(probing_metrics, split, ["exist", "auprc"])
    row["Finetuned AUPRC"] = format_metric(finetune_metrics, split, ["exist", "auprc"])
    row["Probing AUROC"] = format_metric(probing_metrics, split, ["exist", "auroc"])
    row["Finetuned AUROC"] = format_metric(finetune_metrics, split, ["exist", "auroc"])
    
    # Edge Weight
    row["Probing W-MAE"] = format_metric(probing_metrics, split, ["weight", "mae"])
    row["Finetuned W-MAE"] = format_metric(finetune_metrics, split, ["weight", "mae"])
    
    # Node TVL
    row["Probing N-MAE"] = format_metric(probing_metrics, split, ["node", "mae"])
    row["Finetuned N-MAE"] = format_metric(finetune_metrics, split, ["node", "mae"])
    
    # Recall@K
    row["Probing Recall@K"] = format_metric(probing_metrics, split, ["recall_at_k"])
    row["Finetuned Recall@K"] = format_metric(finetune_metrics, split, ["recall_at_k"])
    
    comparison.append(row)

comparison_df = pd.DataFrame(comparison)
print("\n主要指标对比：")
print(comparison_df.to_string(index=False))

print("\n" + "=" * 80)
print("关键发现：")
print("=" * 80)

# 计算提升幅度
try:
    test_probing_auprc = probing_metrics["test"]["exist"]["auprc"]
    test_finetune_auprc = finetune_metrics["test"]["exist"]["auprc"]
    
    if isinstance(test_probing_auprc, (int, float)) and isinstance(test_finetune_auprc, (int, float)):
        improvement = ((test_finetune_auprc - test_probing_auprc) / test_probing_auprc) * 100
        print(f"• Finetuned相比Probing的AUPRC提升: {improvement:+.2f}%")
        print(f"  Probing: {test_probing_auprc:.4f} → Finetuned: {test_finetune_auprc:.4f}")
        
        if improvement > 5:
            print("  → 微调带来显著提升，预训练表示需要适应新任务")
        elif improvement > 0:
            print("  → 微调略有改善，预训练表示基本可用")
        else:
            print("  → 微调未改善，可能存在过拟合或数据不足")
except:
    print("• 无法计算提升幅度（请检查metrics格式）")

print("\n建议：")
print("1. 如果Finetuned显著优于Probing → 需要微调，考虑增加数据或调整学习率")
print("2. 如果Probing接近Finetuned → 预训练表示很好，可以考虑节省计算成本")
print("3. 如果两者都不理想 → 考虑调整负采样比例、增加训练轮数或修改模型结构")

In [ ]:
# 训练曲线可视化
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Frozen-Encoder Probing - Training Losses
if probing_history["train"]:
    epochs_probing = range(1, len(probing_history["train"]) + 1)
    exist_losses = [h["exist_loss"] for h in probing_history["train"]]
    weight_losses = [h["weight_loss"] for h in probing_history["train"]]
    node_losses = [h["node_loss"] for h in probing_history["train"]]
    
    axes[0, 0].plot(epochs_probing, exist_losses, label="Exist Loss", marker="o")
    axes[0, 0].plot(epochs_probing, weight_losses, label="Weight Loss", marker="s")
    axes[0, 0].plot(epochs_probing, node_losses, label="Node Loss", marker="^")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].set_title("Frozen-Encoder Probing: Training Losses")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

# 2. Finetuned - Training Losses
if finetune_history["train"]:
    epochs_finetune = range(1, len(finetune_history["train"]) + 1)
    exist_losses = [h["exist_loss"] for h in finetune_history["train"]]
    weight_losses = [h["weight_loss"] for h in finetune_history["train"]]
    node_losses = [h["node_loss"] for h in finetune_history["train"]]
    
    axes[0, 1].plot(epochs_finetune, exist_losses, label="Exist Loss", marker="o")
    axes[0, 1].plot(epochs_finetune, weight_losses, label="Weight Loss", marker="s")
    axes[0, 1].plot(epochs_finetune, node_losses, label="Node Loss", marker="^")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].set_title("Finetuned Encoder: Training Losses")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# 3. Validation AUPRC Comparison
if probing_history["val"] and finetune_history["val"]:
    epochs_probing = range(1, len(probing_history["val"]) + 1)
    probing_auprc = [h.get("exist", {}).get("auprc", 0) for h in probing_history["val"]]
    
    epochs_finetune = range(1, len(finetune_history["val"]) + 1)
    finetune_auprc = [h.get("exist", {}).get("auprc", 0) for h in finetune_history["val"]]
    
    axes[1, 0].plot(epochs_probing, probing_auprc, label="Probing", marker="o", linewidth=2)
    axes[1, 0].plot(epochs_finetune, finetune_auprc, label="Finetuned", marker="s", linewidth=2)
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Validation AUPRC")
    axes[1, 0].set_title("Validation AUPRC Over Time")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# 4. Validation Weight MAE Comparison
if probing_history["val"] and finetune_history["val"]:
    probing_mae = [h.get("weight", {}).get("mae", 0) for h in probing_history["val"]]
    finetune_mae = [h.get("weight", {}).get("mae", 0) for h in finetune_history["val"]]
    
    axes[1, 1].plot(epochs_probing, probing_mae, label="Probing", marker="o", linewidth=2)
    axes[1, 1].plot(epochs_finetune, finetune_mae, label="Finetuned", marker="s", linewidth=2)
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Validation Weight MAE")
    axes[1, 1].set_title("Validation Weight MAE Over Time")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ 训练曲线已保存到:", Path(OUTPUT_DIR) / "training_curves.png")

## 保存实验结果

生成符合 `experiment-plan.md` 规范的输出文件：

### 1. data_quality.json
每个snapshot的数据质量统计：
- N_nodes, N_edges（节点数和边数）
- pct_target_null_dropped（目标为null的边比例）
- pct_endpoint_missing_dropped（端点缺失的边比例）
- overlap_ratio_next_week（与下一周的节点重叠率）
- pct_category_unknown（category缺失比例）

### 2. metrics.json
完整的实验结果：
- Frozen-encoder probing的train/val/test指标
- Finetuned encoder的train/val/test指标
- 配置参数（便于复现）

### 3. predictions_edges_test.csv
测试集每条边的预测结果：
- 时间戳（t和t+1）
- 节点ID（u和v）
- 真实标签vs预测标签
- 边权（真实vs预测）

### 4. predictions_nodes_test.csv
测试集每个节点的预测结果：
- 时间戳
- 节点ID
- TVL变化（真实vs预测）
- 当前TVL和category

In [ ]:
def predict_samples(
    model: GraphPFNLinkPredictor,
    pairs: List[WeekPair],
    device: torch.device,
    edge_batch_size: Optional[int],
) -> List[Dict]:
    model.eval()
    outputs = []

    with torch.no_grad():
        for sample in pairs:
            graph = dgl.graph((sample.edge_src_t, sample.edge_dst_t), num_nodes=len(sample.node_ids))
            features = torch.tensor(sample.features_t, dtype=torch.float32)
            h = model.encode(graph, features, device=device)

            node_pred = model.node_head(h).cpu().numpy()

            src = torch.tensor(sample.pair_src, dtype=torch.long, device=device)
            dst = torch.tensor(sample.pair_dst, dtype=torch.long, device=device)

            all_logits = []
            all_weight = []

            if edge_batch_size is None:
                logits, w_pred = model.link_scorer(h, src, dst)
                all_logits.append(logits.cpu())
                all_weight.append(w_pred.cpu())
            else:
                for batch_idx in iter_edge_batches(len(src), edge_batch_size, rng=None, shuffle=False):
                    logits, w_pred = model.link_scorer(h, src[batch_idx], dst[batch_idx])
                    all_logits.append(logits.cpu())
                    all_weight.append(w_pred.cpu())

            logits = torch.cat(all_logits).numpy()
            weight_pred = torch.cat(all_weight).numpy()

            outputs.append(
                {
                    "time_t": sample.time_t,
                    "time_t1": sample.time_t1,
                    "pair_src": sample.pair_src,
                    "pair_dst": sample.pair_dst,
                    "y_exist": sample.y_exist,
                    "y_weight": sample.y_weight,
                    "weight_mask": sample.weight_mask,
                    "node_ids": sample.node_ids,
                    "categories": sample.categories,
                    "sizes_t": sample.sizes_t,
                    "y_node": sample.y_node,
                    "node_mask": sample.node_mask,
                    "exist_logits": logits,
                    "weight_pred": weight_pred,
                    "node_pred": node_pred,
                }
            )

    return outputs

## 经济学分析（最低要求的三张图）

根据 `experiment-plan.md` 第9节，生成以下可视化：

### 1. AUPRC over time
- 目的：评估预测稳定性
- 绘制：每个test pair的AUPRC随时间变化
- 预期：模型在危机期间可能性能下降

### 2. Crisis event study
- 目的：分析危机期间的系统性风险指标
- 事件窗口：
  - Terra崩溃：2022-04-01 至 2022-06-30
  - FTX崩溃：2022-10-01 至 2022-12-31
- 指标：
  - **Concentration**: 最大边占系统总敞口的比例
  - **Max counterparty exposure**: 最大对手方敞口（按sector聚合）

### 3. Sector connectivity heatmap
- 目的：比较真实vs预测的部门间敞口矩阵
- X轴：source sector（发起方部门）
- Y轴：target sector（接收方部门）
- 颜色：log1p(总敞口)
- 左图：真实值，右图：预测值

## 预测函数

对给定的一组周对样本进行批量预测：

**流程**：
1. 对每个样本构建图并提取节点embeddings
2. 使用Link Scorer预测所有pairs的existence和weight
3. 使用Node Head预测所有节点的TVL变化
4. 返回预测结果用于评估

**特点**：
- 支持edge batch处理（节省内存）
- 无需梯度计算（推理模式）
- 返回原始logits（便于后续分析）

In [ ]:
def compute_exist_metrics(y_true: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    metrics = {}
    if len(np.unique(y_true)) > 1:
        metrics["auprc"] = float(average_precision_score(y_true, y_score))
        metrics["auroc"] = float(roc_auc_score(y_true, y_score))
    else:
        metrics["auprc"] = float("nan")
        metrics["auroc"] = float("nan")
    return metrics


def compute_regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    return {"mae": float(mae), "rmse": float(rmse)}


def weighted_mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    weights = np.maximum(y_true, 0.0)
    denom = weights.sum()
    if denom <= 0:
        return float("nan")
    return float(np.sum(np.abs(y_true - y_pred) * weights) / denom)


def recall_at_k(y_true_exist: np.ndarray, y_true_weight: np.ndarray, y_score: np.ndarray, k: int) -> float:
    pos_mask = y_true_exist > 0.5
    if pos_mask.sum() == 0:
        return float("nan")

    topk_true = np.argsort(y_true_weight[pos_mask])[-k:]
    true_indices = np.flatnonzero(pos_mask)[topk_true]

    topk_pred = np.argsort(y_score)[-k:]
    hit = len(set(true_indices) & set(topk_pred))
    return hit / float(k)

## 总结与说明

### 实验设计
本notebook完整实现了 `experiment-plan.md` 规范的实验：

✅ **预测任务**
- Edge existence分类（主要任务）
- Edge weight回归（主要任务，仅正例）
- Node TVL log-change回归（辅助任务）

✅ **模型对比**
- **Frozen-encoder probing**: GraphPFN encoder冻结，仅训练scorer
  - 代表"zero-shot"或"linear probing"场景
  - 测试预训练表示是否足够好
- **Finetuned encoder**: 端到端微调整个模型
  - 代表"fully finetuned"场景
  - 测试适应新任务的性能上限

✅ **数据处理**
- Uniform negative sampling (neg:pos = 5:1)
- 固定随机种子（可复现）
- 时间序列切分（避免信息泄漏）
- 完整的数据质量追踪

✅ **评估指标**
- AUPRC/AUROC（edge existence）
- MAE/RMSE（edge weight & node TVL）
- Recall@K（系统性风险）
- Weighted MAE（重视大边的误差）

✅ **输出文件**
- data_quality.json（数据统计）
- metrics.json（实验结果）
- predictions_edges_test.csv（边预测）
- predictions_nodes_test.csv（节点预测）

### 关键实现细节
1. **Multi-task learning**: 三个loss同时优化（1:1:0.5权重）
2. **Edge batching**: 支持大规模图的分batch训练
3. **Early stopping**: 基于验证集AUPRC，patience=3
4. **Encoder mode控制**: 冻结时确保encoder在eval mode

### 使用建议
- **快速测试**: 设置 `MAX_WEEKS=10` 和 `EDGE_BATCH_SIZE=1000`
- **完整实验**: 设置 `MAX_WEEKS=None` 使用全部数据
- **微调encoder**: 设置 `FINETUNE_FULL=True`
- **调整负采样**: 修改 `NEG_RATIO`（默认5:1）

### 下一步
1. 检查metrics.json对比两种方法的性能
2. 分析predictions CSV中的具体案例
3. 根据经济学分析图表识别系统性风险模式
4. 考虑ablation study（如移除node loss）

In [ ]:
def evaluate_predictions(preds: List[Dict], k: int = 100) -> Dict[str, Dict[str, float]]:
    exist_true = []
    exist_score = []
    weight_true = []
    weight_pred = []
    node_true = []
    node_pred = []
    per_pair_recall = []

    for item in preds:
        y_exist = item["y_exist"]
        y_weight = item["y_weight"]
        weight_mask = item["weight_mask"]
        exist_prob = 1 / (1 + np.exp(-item["exist_logits"]))
        score = exist_prob * item["weight_pred"]

        exist_true.append(y_exist)
        exist_score.append(exist_prob)

        if weight_mask.sum() > 0:
            weight_true.append(y_weight[weight_mask > 0.5])
            weight_pred.append(item["weight_pred"][weight_mask > 0.5])

        node_mask = item["node_mask"]
        if node_mask.any():
            node_true.append(item["y_node"][node_mask])
            node_pred.append(item["node_pred"][node_mask])

        per_pair_recall.append(recall_at_k(y_exist, y_weight, score, k=k))

    exist_true = np.concatenate(exist_true)
    exist_score = np.concatenate(exist_score)

    exist_metrics = compute_exist_metrics(exist_true, exist_score)

    weight_metrics = {"mae": float("nan"), "rmse": float("nan"), "weighted_mae": float("nan")}
    if weight_true:
        weight_true = np.concatenate(weight_true)
        weight_pred = np.concatenate(weight_pred)
        weight_metrics = compute_regression_metrics(weight_true, weight_pred)
        weight_metrics["weighted_mae"] = weighted_mae(weight_true, weight_pred)

    node_metrics = {"mae": float("nan"), "rmse": float("nan")}
    if node_true:
        node_true = np.concatenate(node_true)
        node_pred = np.concatenate(node_pred)
        node_metrics = compute_regression_metrics(node_true, node_pred)

    recall_k = float(np.nanmean(per_pair_recall))

    return {
        "exist": exist_metrics,
        "weight": weight_metrics,
        "node": node_metrics,
        "recall_at_k": recall_k,
    }

## 评估主函数

聚合所有样本的预测结果，计算综合指标。

**流程**：
1. 收集所有样本的predictions和labels
2. 计算edge-level指标（AUPRC, AUROC, MAE, RMSE）
3. 计算node-level指标（MAE, RMSE）
4. 计算Recall@K（系统性风险指标）
5. 返回结构化的metrics字典

## Training: Frozen-Encoder Probing

## 训练工具函数

包含带早停机制的训练循环和最佳模型保存功能。

In [ ]:
def train_with_early_stopping(
    model: GraphPFNLinkPredictor,
    train_pairs: List[WeekPair],
    val_pairs: List[WeekPair],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    edge_batch_size: Optional[int],
    finetune_encoder: bool,
    epochs: int,
    patience: int = 3,
    metric_key: str = "auprc",
    model_path: Optional[str] = None,
) -> Tuple[GraphPFNLinkPredictor, Dict]:
    """
    带早停机制的训练函数
    
    Args:
        model: 模型实例
        train_pairs: 训练数据
        val_pairs: 验证数据
        optimizer: 优化器
        device: 设备
        edge_batch_size: edge batch大小
        finetune_encoder: 是否微调encoder
        epochs: 最大训练轮数
        patience: 早停耐心值（连续多少轮无改善则停止）
        metric_key: 用于早停的指标（"auprc" 或 "auroc"）
        model_path: 最佳模型保存路径（None则不保存）
    
    Returns:
        训练好的模型和训练历史
    """
    best_metric = -float("inf")
    no_improve_count = 0
    best_epoch = 0
    history = {"train": [], "val": []}
    
    for epoch in range(1, epochs + 1):
        # 训练
        train_losses = train_one_epoch(
            model,
            train_pairs,
            optimizer,
            device=device,
            edge_batch_size=edge_batch_size,
            finetune_encoder=finetune_encoder,
        )
        
        # 验证
        model.eval()
        val_preds = predict_samples(model, val_pairs, device, edge_batch_size)
        val_metrics = evaluate_predictions(val_preds)
        
        current_metric = val_metrics["exist"].get(metric_key, 0.0)
        if isinstance(current_metric, str):
            current_metric = 0.0
        
        history["train"].append(train_losses)
        history["val"].append(val_metrics)
        
        print(f"Epoch {epoch}: train={train_losses}")
        print(f"  Val: exist={val_metrics['exist']}, weight={val_metrics['weight']}, node={val_metrics['node']}")
        print(f"  Val {metric_key}={current_metric:.4f}")
        
        # 早停检查
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            no_improve_count = 0
            
            # 保存最佳模型
            if model_path is not None:
                torch.save(model.state_dict(), model_path)
                print(f"  ✓ New best model saved (epoch {epoch}, {metric_key}={best_metric:.4f})")
        else:
            no_improve_count += 1
            print(f"  No improvement ({no_improve_count}/{patience})")
        
        if no_improve_count >= patience:
            print(f"\nEarly stopping at epoch {epoch} (best epoch: {best_epoch}, best {metric_key}: {best_metric:.4f})")
            break
    
    # 加载最佳模型
    if model_path is not None and Path(model_path).exists():
        model.load_state_dict(torch.load(model_path))
        print(f"Loaded best model from epoch {best_epoch}")
    
    return model, history

In [ ]:
# 创建frozen-encoder probing模型
probing_model = GraphPFNLinkPredictor(encoder=encoder_frozen, embed_dim=embed_dim, hidden_dim=HIDDEN_DIM).to(DEVICE)
set_encoder_trainable(probing_model.encoder, trainable=False)

optimizer = torch.optim.Adam(
    [p for p in probing_model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

print("=" * 60)
print("训练 Frozen-Encoder Probing 模型")
print("=" * 60)
print("说明：GraphPFN encoder权重冻结，仅训练Link Scorer和Node Head")
print("这相当于测试预训练表示是否可以直接线性分离/回归任务\n")

probing_model, probing_history = train_with_early_stopping(
    probing_model,
    train_pairs,
    val_pairs,
    optimizer,
    device=DEVICE,
    edge_batch_size=EDGE_BATCH_SIZE,
    finetune_encoder=False,
    epochs=EPOCHS,
    patience=3,
    metric_key="auprc",
    model_path=str(Path(OUTPUT_DIR) / "best_probing_model.ckpt"),
)

print("\n" + "=" * 60)
print("训练完成！")
print("=" * 60)

## Training: Finetuned Encoder

In [ ]:
# 创建finetuned encoder模型
finetune_model = GraphPFNLinkPredictor(encoder=encoder_finetune, embed_dim=embed_dim, hidden_dim=HIDDEN_DIM).to(DEVICE)
if FINETUNE_FULL:
    set_encoder_trainable(finetune_model.encoder, trainable=True)
    print("配置：完整微调GraphPFN encoder")
else:
    print(f"配置：仅训练scorer heads（encoder仍冻结，设置FINETUNE_FULL=True来微调）")

optimizer_ft = torch.optim.Adam(
    [p for p in finetune_model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

print("\n" + "=" * 60)
print("训练 Finetuned Encoder 模型")
print("=" * 60)
print("说明：根据FINETUNE_FULL配置决定是否更新GraphPFN encoder")
print("端到端训练整个模型（encoder + scorer + node head）\n")

finetune_model, finetune_history = train_with_early_stopping(
    finetune_model,
    train_pairs,
    val_pairs,
    optimizer_ft,
    device=DEVICE,
    edge_batch_size=EDGE_BATCH_SIZE,
    finetune_encoder=True,
    epochs=EPOCHS,
    patience=3,
    metric_key="auprc",
    model_path=str(Path(OUTPUT_DIR) / "best_finetune_model.ckpt"),
)

print("\n" + "=" * 60)
print("训练完成！")
print("=" * 60)

from datetime import datetime
from pathlib import Path

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== 1. data_quality.json =====
# 符合experiment-plan.md规范的字段名：N_nodes, N_edges
quality = {
    "snapshots": [
        {
            "date": s["date"],
            "N_nodes": len(s["node_ids"]),  # 规范要求的字段名
            "N_edges": int(s["num_edges"]),  # 规范要求的字段名
            "pct_target_null_dropped": s["pct_target_null_dropped"],
            "pct_endpoint_missing_dropped": s["pct_endpoint_missing_dropped"],
            "overlap_ratio_next_week": s.get("overlap_ratio_next_week"),
            "pct_category_unknown": s["pct_category_unknown"],
        }
        for s in snapshots
    ],
    "generated_at": datetime.now().isoformat(),
}

with open(Path(OUTPUT_DIR) / "data_quality.json", "w") as f:
    json.dump(quality, f, indent=2)
print("✓ 已保存 data_quality.json")

# ===== 2. metrics.json =====
metrics_payload = {
    "frozen_encoder_probing": probing_metrics,
    "finetuned": finetune_metrics,
    "config": {
        "neg_ratio": NEG_RATIO,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "edge_batch_size": EDGE_BATCH_SIZE,
        "hidden_dim": HIDDEN_DIM,
        "epochs": EPOCHS,
        "seed": SEED,
        "finetune_full": FINETUNE_FULL,
        "loss_weights": {
            "exist": EXIST_LOSS_WEIGHT,
            "weight": WEIGHT_LOSS_WEIGHT,
            "node": NODE_LOSS_WEIGHT,
        }
    },
    "generated_at": datetime.now().isoformat(),
}

with open(Path(OUTPUT_DIR) / "metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=2)
print("✓ 已保存 metrics.json")

# ===== 3. predictions_edges_test.csv (finetuned模型) =====
edge_rows = []
for item in finetune_preds_test:
    exist_prob = 1 / (1 + np.exp(-item["exist_logits"]))
    for i in range(len(item["pair_src"])):
        u = item["node_ids"][int(item["pair_src"][i])]
        v = item["node_ids"][int(item["pair_dst"][i])]
        edge_rows.append(
            {
                "time_t": item["time_t"],
                "time_t1": item["time_t1"],
                "u_id": u,
                "v_id": v,
                "y_exist_true": float(item["y_exist"][i]),
                "y_exist_pred": float(exist_prob[i]),
                "y_w_true": float(item["y_weight"][i]),
                "y_w_pred": float(item["weight_pred"][i]),
                "is_positive": int(item["y_exist"][i] > 0.5),
            }
        )

edges_df = pd.DataFrame(edge_rows)
edges_df.to_csv(Path(OUTPUT_DIR) / "predictions_edges_test.csv", index=False)
print(f"✓ 已保存 predictions_edges_test.csv ({len(edge_rows)} 行)")

# ===== 4. predictions_nodes_test.csv (finetuned模型) =====
node_rows = []
for item in finetune_preds_test:
    for i, node_id in enumerate(item["node_ids"]):
        if not item["node_mask"][i]:
            continue
        node_rows.append(
            {
                "time_t": item["time_t"],
                "time_t1": item["time_t1"],
                "node_id": node_id,
                "y_node_true": float(item["y_node"][i]),
                "y_node_pred": float(item["node_pred"][i]),
                "size_t": float(item["sizes_t"][i]),
                "category": item["categories"][i],
            }
        )

nodes_df = pd.DataFrame(node_rows)
nodes_df.to_csv(Path(OUTPUT_DIR) / "predictions_nodes_test.csv", index=False)
print(f"✓ 已保存 predictions_nodes_test.csv ({len(node_rows)} 行)")

print("\n" + "=" * 60)
print(f"所有输出文件已保存到: {OUTPUT_DIR}")
print("=" * 60)

In [ ]:
probing_preds_train = predict_samples(probing_model, train_pairs, DEVICE, EDGE_BATCH_SIZE)
probing_preds_val = predict_samples(probing_model, val_pairs, DEVICE, EDGE_BATCH_SIZE)
probing_preds_test = predict_samples(probing_model, test_pairs, DEVICE, EDGE_BATCH_SIZE)

probing_metrics = {
    "train": evaluate_predictions(probing_preds_train),
    "val": evaluate_predictions(probing_preds_val),
    "test": evaluate_predictions(probing_preds_test),
}

finetune_preds_train = predict_samples(finetune_model, train_pairs, DEVICE, EDGE_BATCH_SIZE)
finetune_preds_val = predict_samples(finetune_model, val_pairs, DEVICE, EDGE_BATCH_SIZE)
finetune_preds_test = predict_samples(finetune_model, test_pairs, DEVICE, EDGE_BATCH_SIZE)

finetune_metrics = {
    "train": evaluate_predictions(finetune_preds_train),
    "val": evaluate_predictions(finetune_preds_val),
    "test": evaluate_predictions(finetune_preds_test),
}

print("Frozen encoder test metrics:", probing_metrics["test"])
print("Finetuned encoder test metrics:", finetune_metrics["test"])

## Save Outputs

In [ ]:
from datetime import datetime

os.makedirs(OUTPUT_DIR, exist_ok=True)

# data_quality.json
quality = {
    "snapshots": [
        {
            "date": s["date"],
            "num_nodes": len(s["node_ids"]),
            "num_edges": int(s["num_edges"]),
            "pct_target_null_dropped": s["pct_target_null_dropped"],
            "pct_endpoint_missing_dropped": s["pct_endpoint_missing_dropped"],
            "overlap_ratio_next_week": s.get("overlap_ratio_next_week"),
            "pct_category_unknown": s["pct_category_unknown"],
        }
        for s in snapshots
    ],
    "generated_at": datetime.now().isoformat(),
}

with open(Path(OUTPUT_DIR) / "data_quality.json", "w") as f:
    json.dump(quality, f, indent=2)

# metrics.json
metrics_payload = {
    "frozen_encoder_probing": probing_metrics,
    "finetuned": finetune_metrics,
    "config": {
        "neg_ratio": NEG_RATIO,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "edge_batch_size": EDGE_BATCH_SIZE,
        "hidden_dim": HIDDEN_DIM,
        "epochs": EPOCHS,
        "seed": SEED,
        "finetune_full": FINETUNE_FULL,
    },
    "generated_at": datetime.now().isoformat(),
}

with open(Path(OUTPUT_DIR) / "metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=2)

# predictions_edges_test.csv (finetuned)
edge_rows = []
for item in finetune_preds_test:
    exist_prob = 1 / (1 + np.exp(-item["exist_logits"]))
    for i in range(len(item["pair_src"])):
        u = item["node_ids"][int(item["pair_src"][i])]
        v = item["node_ids"][int(item["pair_dst"][i])]
        edge_rows.append(
            {
                "time_t": item["time_t"],
                "time_t1": item["time_t1"],
                "u_id": u,
                "v_id": v,
                "y_exist_true": float(item["y_exist"][i]),
                "y_exist_pred": float(exist_prob[i]),
                "y_w_true": float(item["y_weight"][i]),
                "y_w_pred": float(item["weight_pred"][i]),
                "is_positive": int(item["y_exist"][i] > 0.5),
            }
        )

edges_df = pd.DataFrame(edge_rows)
edges_df.to_csv(Path(OUTPUT_DIR) / "predictions_edges_test.csv", index=False)

# predictions_nodes_test.csv (finetuned)
node_rows = []
for item in finetune_preds_test:
    for i, node_id in enumerate(item["node_ids"]):
        if not item["node_mask"][i]:
            continue
        node_rows.append(
            {
                "time_t": item["time_t"],
                "time_t1": item["time_t1"],
                "node_id": node_id,
                "y_node_true": float(item["y_node"][i]),
                "y_node_pred": float(item["node_pred"][i]),
                "size_t": float(item["sizes_t"][i]),
                "category": item["categories"][i],
            }
        )

nodes_df = pd.DataFrame(node_rows)
nodes_df.to_csv(Path(OUTPUT_DIR) / "predictions_nodes_test.csv", index=False)

print("Saved outputs to", OUTPUT_DIR)

## Economic Analysis (Minimum Required Plots)

In [ ]:
# 1) AUPRC over time (per week-pair)
auprc_series = []
for item in finetune_preds_test:
    y_true = item["y_exist"]
    y_score = 1 / (1 + np.exp(-item["exist_logits"]))
    if len(np.unique(y_true)) > 1:
        auprc = average_precision_score(y_true, y_score)
    else:
        auprc = float("nan")
    auprc_series.append({"time_t": item["time_t"], "time_t1": item["time_t1"], "auprc": auprc})

auprc_df = pd.DataFrame(auprc_series)
plt.figure(figsize=(10, 4))
plt.plot(auprc_df["time_t1"], auprc_df["auprc"], marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("AUPRC Over Time (Test Pairs)")
plt.tight_layout()
plt.show()

In [ ]:
# 2) Crisis event study (Terra / FTX windows)
EVENT_WINDOWS = {
    "Terra": ("2022-04-01", "2022-06-30"),
    "FTX": ("2022-10-01", "2022-12-31"),
}

metrics_over_time = []
for snap in snapshots:
    total_weight = float(np.sum(snap["edge_weight"]))
    max_edge = float(np.max(snap["edge_weight"])) if len(snap["edge_weight"]) else 0.0
    concentration = max_edge / max(total_weight, 1e-9)

    outflow = {}
    for src_idx, w in zip(snap["edge_src"], snap["edge_weight"]):
        cat = snap["categories"][int(src_idx)]
        outflow[cat] = outflow.get(cat, 0.0) + float(w)
    max_counterparty = max(outflow.values()) if outflow else 0.0

    metrics_over_time.append({
        "date": snap["date"],
        "concentration": concentration,
        "max_counterparty_exposure": max_counterparty,
    })

metrics_df = pd.DataFrame(metrics_over_time)
metrics_df["date"] = pd.to_datetime(metrics_df["date"])

plt.figure(figsize=(10, 4))
plt.plot(metrics_df["date"], metrics_df["concentration"], label="Concentration")
plt.plot(metrics_df["date"], metrics_df["max_counterparty_exposure"], label="Max Counterparty Exposure")
for label, (start, end) in EVENT_WINDOWS.items():
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    plt.axvspan(start_dt, end_dt, alpha=0.2, label=label)
plt.legend()
plt.title("Crisis Event Study")
plt.tight_layout()
plt.show()

In [ ]:
# 3) Sector connectivity heatmap (true vs predicted, test split)

def build_category_matrix(node_categories, pair_src, pair_dst, weights):
    cats = sorted(set(node_categories))
    idx_map = {c: i for i, c in enumerate(cats)}
    mat = np.zeros((len(cats), len(cats)), dtype=np.float64)
    for s, d, w in zip(pair_src, pair_dst, weights):
        c_s = node_categories[int(s)]
        c_d = node_categories[int(d)]
        mat[idx_map[c_s], idx_map[c_d]] += float(w)
    return mat, cats

# Aggregate over test week pairs
true_mats = []
pred_mats = []
cat_labels = None

for item in finetune_preds_test:
    exist_prob = 1 / (1 + np.exp(-item["exist_logits"]))
    pred_weight = exist_prob * item["weight_pred"]

    mat_true, cats = build_category_matrix(
        item["categories"],
        item["pair_src"],
        item["pair_dst"],
        item["y_weight"],
    )
    mat_pred, _ = build_category_matrix(
        item["categories"],
        item["pair_src"],
        item["pair_dst"],
        pred_weight,
    )
    true_mats.append(mat_true)
    pred_mats.append(mat_pred)
    cat_labels = cats

if true_mats:
    true_avg = np.mean(true_mats, axis=0)
    pred_avg = np.mean(pred_mats, axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(true_avg, ax=axes[0], cmap="Blues", xticklabels=cat_labels, yticklabels=cat_labels)
    axes[0].set_title("True Sector Connectivity")
    sns.heatmap(pred_avg, ax=axes[1], cmap="Reds", xticklabels=cat_labels, yticklabels=cat_labels)
    axes[1].set_title("Predicted Sector Connectivity")
    plt.tight_layout()
    plt.show()

## Notes
- Zero-shot here is implemented as frozen-encoder probing (encoder frozen, scorer trained).
- Finetuned updates the encoder (optionally full or adapter-only).
- Use `MAX_WEEKS` and `EDGE_BATCH_SIZE` to scale compute on large datasets.